## Scenario: MLflow with Azure Machine Learning

**Architecture:**
- Tracking: Azure ML Workspace (managed MLflow)
- Backend: Azure ML managed storage
- Artifacts: Azure ML workspace storage (automatic)

**Setup:**
1. Run `setup-azureml-workspace.sh` to create workspace
2. Run `az login` to authenticate
3. Install packages: `pip install azure-ai-ml azure-identity azureml-mlflow`
4. Fill in `.env` file with workspace details
5. Run cells in order


### Install Required Packages


In [1]:
# Install Azure ML packages
import subprocess, sys

packages = ['azure-ai-ml', 'azure-identity', 'azureml-mlflow']
for pkg in packages:
    try:
        if pkg == 'azure-ai-ml':
            import azure.ai.ml
        elif pkg == 'azure-identity':
            import azure.identity
        elif pkg == 'azureml-mlflow':
            import azureml.mlflow
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"✓ Installed {pkg}")


### Load Configuration


In [2]:
import os

# Load .env file
try:
    from dotenv import load_dotenv
    load_dotenv('.env')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])
    from dotenv import load_dotenv
    load_dotenv('.env')

# Get workspace configuration
subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID', '')
resource_group = os.environ.get('AZURE_RESOURCE_GROUP', 'rg-azureml-mlflow')
workspace_name = os.environ.get('AZURE_WORKSPACE_NAME', '')

if not subscription_id or 'your_' in subscription_id:
    print("⚠️ Set AZURE_SUBSCRIPTION_ID in .env file")
    print("   Get it with: az account show --query id -o tsv")
elif not workspace_name or 'your_' in workspace_name:
    print("⚠️ Set AZURE_WORKSPACE_NAME in .env file")
else:
    print(f"✓ Configuration loaded")
    print(f"  Subscription: {subscription_id[:8]}...")
    print(f"  Resource Group: {resource_group}")
    print(f"  Workspace: {workspace_name}")


✓ Configuration loaded
  Subscription: a23fa87c...
  Resource Group: rg-azureml-mlflow
  Workspace: aml-mlflow-8009


### Connect to Azure ML Workspace


In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow

# Authenticate and connect to workspace
credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

# Get MLflow tracking URI (automatically configured!)
workspace = ml_client.workspaces.get(workspace_name)
mlflow_tracking_uri = workspace.mlflow_tracking_uri

# Set MLflow tracking URI
mlflow.set_tracking_uri(mlflow_tracking_uri)

print(f"✓ Connected to Azure ML Workspace: {workspace_name}")
print(f"✓ MLflow Tracking URI: {mlflow_tracking_uri}")


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✓ Connected to Azure ML Workspace: aml-mlflow-8009
✓ MLflow Tracking URI: azureml://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009


### List Experiments


In [4]:
# List all experiments
experiments = mlflow.search_experiments(view_type=mlflow.entities.ViewType.ACTIVE_ONLY)
print(f"✓ Found {len(experiments)} experiments")
experiments

✓ Found 0 experiments


[]

### Train and Log Model


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("iris-classification")

with mlflow.start_run():
    X, y = load_iris(return_X_y=True)
    
    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)
    
    lr = LogisticRegression(**params).fit(X, y)
    mlflow.log_metric("accuracy", accuracy_score(y, lr.predict(X)))
    
    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"✓ Model logged")
    print(f"  Artifacts URI: {mlflow.get_artifact_uri()}")
    print(f"  Run URI: {mlflow.get_tracking_uri()}/#/experiments/{mlflow.active_run().info.experiment_id}/runs/{mlflow.active_run().info.run_id}")


2025/11/02 10:17:10 INFO mlflow.tracking.fluent: Experiment with name 'iris-classification' does not exist. Creating a new experiment.
2025/11/02 10:17:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ Model logged
  Artifacts URI: azureml://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009/experiments/db774d9a-4a13-4f3c-b8f2-bda89378540e/runs/cceeb062-ccea-4ab8-90db-f4af3522e211/artifacts
  Run URI: azureml://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009/#/experiments/db774d9a-4a13-4f3c-b8f2-bda89378540e/runs/cceeb062-ccea-4ab8-90db-f4af3522e211
🏃 View run quiet_bread_j1kvrns0 at: https://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009/#/experiments/db774d9a-4a13-4f3c-b8f2-bda89378540e/runs/cceeb062-ccea-4ab8-90db-f4af3522e211
🧪 View experiment 

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("iris-classification")

with mlflow.start_run():
    X, y = load_iris(return_X_y=True)
    
    params = {"n_estimators": 100, "max_depth": 5, "random_state": 42}
    mlflow.log_params(params)
    
    rf = RandomForestClassifier(**params).fit(X, y)
    mlflow.log_metric("accuracy", accuracy_score(y, rf.predict(X)))
    
    mlflow.sklearn.log_model(rf, artifact_path="models")
    print(f"✓ Model logged")
    print(f"  Artifacts URI: {mlflow.get_artifact_uri()}")
    print(f"  Run URI: {mlflow.get_tracking_uri()}/#/experiments/{mlflow.active_run().info.experiment_id}/runs/{mlflow.active_run().info.run_id}")


2025/11/02 10:17:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✓ Model logged
  Artifacts URI: azureml://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009/experiments/db774d9a-4a13-4f3c-b8f2-bda89378540e/runs/e3eeece9-f581-4ce0-a540-1f91d9e8c8b5/artifacts
  Run URI: azureml://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009/#/experiments/db774d9a-4a13-4f3c-b8f2-bda89378540e/runs/e3eeece9-f581-4ce0-a540-1f91d9e8c8b5
🏃 View run plucky_tray_spcm1dgh at: https://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/a23fa87c-802c-4fdf-9e59-e3d7969bcf31/resourceGroups/rg-azureml-mlflow/providers/Microsoft.MachineLearningServices/workspaces/aml-mlflow-8009/#/experiments/db774d9a-4a13-4f3c-b8f2-bda89378540e/runs/e3eeece9-f581-4ce0-a540-1f91d9e8c8b5
🧪 View experiment 

### Model Registry


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())
client.search_registered_models()


In [ ]:
# Get latest run and register model
runs = mlflow.search_runs(experiment_ids=["0"], max_results=1)
if len(runs) > 0:
    run_id = runs.iloc[0].run_id
    print(f"✓ Latest run: {run_id}")
else:
    print("No runs found")


In [ ]:
# Register the model
if 'run_id' in locals():
    mlflow.register_model(
        model_uri=f"runs:/{run_id}/models",
        name="iris-classifier-azureml"
    )
    print("✓ Model registered in Azure ML Model Registry")


### Troubleshooting

**Authentication Issues:**
- Run `az login` to authenticate with Azure CLI
- Or set Service Principal credentials in `.env`

**Workspace Not Found:**
- Verify workspace name: `az ml workspace list -g <resource-group>`
- Check subscription: `az account show`

**MLflow Tracking URI Issues:**
- Ensure workspace is created successfully
- MLflow tracking URI is automatically provided by Azure ML

**Cost Management:**
- Workspace itself is free
- You only pay for compute when running experiments
- Delete workspace: `az ml workspace delete -g <rg> -w <workspace>`
